# ToMnet Figure 3 Visualization

This notebook reproduces the visualizations from Figure 3 of the "Machine Theory of Mind" paper.

Figure 3 shows:
- (a) Action likelihood vs number of past observations
- (b) 2D character embeddings colored by most frequent action
- (c) KL-divergence matrix for cross-species generalization
- (d) Hierarchical inference on mixed species

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import torch
from collections import defaultdict
import pandas as pd
from sklearn.decomposition import PCA

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Action names for visualization
ACTION_NAMES = ['Up', 'Down', 'Left', 'Right', 'Stay']
ACTION_COLORS = ['red', 'blue', 'green', 'orange', 'purple']

## Load Evaluation Results

In [ ]:
# Load evaluation results
# These should be generated by running the evaluation script
with open('evaluation_results_figure3.pkl', 'rb') as f:
    results = pickle.load(f)

print("Available models:", list(results.keys()))
print("Available datasets for first model:", list(results[list(results.keys())[0]].keys()))

## Figure 3a: Action Likelihood vs Number of Past Observations

In [ ]:
def plot_action_likelihood_vs_n_past(results, alpha_values=[0.01, 3.0]):
    """Plot action likelihood vs number of past observations"""
    fig, axes = plt.subplots(1, len(alpha_values), figsize=(12, 5))
    if len(alpha_values) == 1:
        axes = [axes]
    
    for i, alpha in enumerate(alpha_values):
        ax = axes[i]
        
        # Plot ToMnet results
        for model_name, model_results in results.items():
            if f'alpha_{alpha}' in model_results:
                accuracy_by_n_past = model_results[f'alpha_{alpha}']['accuracy_by_n_past']
                
                n_past_values = sorted(accuracy_by_n_past.keys())
                accuracies = [accuracy_by_n_past[n] for n in n_past_values]
                
                ax.plot(n_past_values, accuracies, 'o-', label=f'ToMnet ({model_name})', linewidth=2)
        
        # Plot Bayes-optimal baseline if available
        if 'bayes_optimal_baseline' in results[list(results.keys())[0]]:
            baseline_data = results[list(results.keys())[0]]['bayes_optimal_baseline']
            
            # Group by n_past
            baseline_by_n_past = defaultdict(list)
            for n_past, accuracy in zip(baseline_data['n_past_values'], baseline_data['action_accuracies']):
                baseline_by_n_past[n_past].append(accuracy)
            
            n_past_values = sorted(baseline_by_n_past.keys())
            baseline_accuracies = [np.mean(baseline_by_n_past[n]) for n in n_past_values]
            
            ax.plot(n_past_values, baseline_accuracies, 's--', label='Bayes-optimal', 
                   color='black', linewidth=2, alpha=0.7)
        
        ax.set_xlabel('Number of past observations')
        ax.set_ylabel('Action likelihood')
        ax.set_title(f'α = {alpha} ({"deterministic" if alpha < 1 else "stochastic"})')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig('figure3a_action_likelihood.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_action_likelihood_vs_n_past(results)

## Figure 3b: 2D Character Embeddings

In [ ]:
def plot_character_embeddings(results, alpha_values=[0.01, 3.0]):
    """Plot 2D character embeddings colored by dominant action"""
    fig, axes = plt.subplots(1, len(alpha_values), figsize=(12, 5))
    if len(alpha_values) == 1:
        axes = [axes]
    
    for i, alpha in enumerate(alpha_values):
        ax = axes[i]
        
        # Get embeddings from first model
        model_name = list(results.keys())[0]
        if f'alpha_{alpha}' in results[model_name]:
            embeddings = results[model_name][f'alpha_{alpha}']['character_embeddings']
            agent_ids = results[model_name][f'alpha_{alpha}']['agent_ids']
            
            # Load original dataset to get agent policies
            try:
                with open(f'data/figure3_alpha_{alpha}.pkl', 'rb') as f:
                    dataset = pickle.load(f)
                
                # Get dominant action for each agent
                agent_actions = {}
                for sample in dataset['data']:
                    agent_id = sample['agent_id']
                    if agent_id not in agent_actions:
                        true_policy = sample['true_policy']
                        dominant_action = np.argmax(true_policy)
                        agent_actions[agent_id] = dominant_action
                
                # Create scatter plot
                for action in range(5):
                    mask = [agent_actions.get(agent_id, 0) == action for agent_id in agent_ids]
                    if np.any(mask):
                        ax.scatter(embeddings[mask, 0], embeddings[mask, 1], 
                                 c=ACTION_COLORS[action], label=ACTION_NAMES[action], 
                                 alpha=0.7, s=50)
                
            except FileNotFoundError:
                print(f"Warning: Could not find dataset file for alpha={alpha}")
                # Fallback: color by agent ID
                scatter = ax.scatter(embeddings[:, 0], embeddings[:, 1], 
                                   c=agent_ids, cmap='tab10', alpha=0.7, s=50)
                plt.colorbar(scatter, ax=ax, label='Agent ID')
        
        ax.set_xlabel('Character embedding dimension 1')
        ax.set_ylabel('Character embedding dimension 2')
        ax.set_title(f'α = {alpha} character embeddings')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('figure3b_character_embeddings.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_character_embeddings(results)

## Figure 3c: KL-Divergence Cross-Species Generalization Matrix

In [ ]:
def plot_kl_divergence_matrix(results, alpha_values=[0.01, 3.0]):
    """Plot KL divergence matrix for cross-species generalization"""
    # Create matrix: rows = training alpha, columns = test alpha
    kl_matrix = np.zeros((len(alpha_values), len(alpha_values)))
    
    for i, train_alpha in enumerate(alpha_values):
        for j, test_alpha in enumerate(alpha_values):
            # Find model trained on train_alpha
            model_key = f'alpha_{train_alpha}' if f'alpha_{train_alpha}' in results else list(results.keys())[0]
            
            if model_key in results and f'alpha_{test_alpha}' in results[model_key]:
                kl_div = results[model_key][f'alpha_{test_alpha}']['mean_kl_divergence']
                kl_matrix[i, j] = kl_div
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(8, 6))
    
    im = ax.imshow(kl_matrix, cmap='viridis', aspect='auto')
    
    # Set ticks and labels
    ax.set_xticks(range(len(alpha_values)))
    ax.set_yticks(range(len(alpha_values)))
    ax.set_xticklabels([f'α={a}' for a in alpha_values])
    ax.set_yticklabels([f'α={a}' for a in alpha_values])
    
    ax.set_xlabel('Test species (α)')
    ax.set_ylabel('Training species (α)')
    ax.set_title('KL divergence: cross-species generalization')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('KL divergence')
    
    # Add text annotations
    for i in range(len(alpha_values)):
        for j in range(len(alpha_values)):
            text = ax.text(j, i, f'{kl_matrix[i, j]:.3f}',
                          ha="center", va="center", color="white" if kl_matrix[i, j] > kl_matrix.max()/2 else "black")
    
    plt.tight_layout()
    plt.savefig('figure3c_kl_divergence_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_kl_divergence_matrix(results)

## Figure 3d: Mixed Species Training

In [ ]:
def plot_mixed_species_results(results):
    """Plot results for mixed species training"""
    if 'mixed' not in results:
        print("Mixed species results not available")
        return
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Performance comparison
    alpha_values = [0.01, 3.0]
    model_types = ['Single α=0.01', 'Single α=3.0', 'Mixed']
    
    performance_data = []
    
    for test_alpha in alpha_values:
        test_key = f'alpha_{test_alpha}'
        
        # Single species models
        single_models = [f'alpha_{a}' for a in alpha_values]
        mixed_model = 'mixed'
        
        test_accuracies = []
        
        for model in single_models + [mixed_model]:
            if model in results and test_key in results[model]:
                acc = results[model][test_key]['action_accuracy']
                test_accuracies.append(acc)
            else:
                test_accuracies.append(0)
        
        performance_data.append(test_accuracies)
    
    # Create grouped bar plot
    x = np.arange(len(alpha_values))
    width = 0.25
    
    for i, model_type in enumerate(model_types):
        accuracies = [performance_data[j][i] for j in range(len(alpha_values))]
        ax1.bar(x + i * width, accuracies, width, label=model_type)
    
    ax1.set_xlabel('Test species')
    ax1.set_ylabel('Action prediction accuracy')
    ax1.set_title('Mixed vs Single Species Training')
    ax1.set_xticks(x + width)
    ax1.set_xticklabels([f'α={a}' for a in alpha_values])
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Character embeddings for mixed model
    if 'mixed' in results and len(alpha_values) >= 2:
        # We need to infer which agents came from which alpha
        # This is a simplified visualization
        mixed_results = results['mixed']
        
        # Try to get embeddings from one of the test sets
        for test_alpha in alpha_values:
            test_key = f'alpha_{test_alpha}'
            if test_key in mixed_results:
                embeddings = mixed_results[test_key]['character_embeddings']
                agent_ids = mixed_results[test_key]['agent_ids']
                
                # Color by agent ID (as proxy for original alpha)
                scatter = ax2.scatter(embeddings[:, 0], embeddings[:, 1], 
                                    c=agent_ids, cmap='viridis', alpha=0.7, s=50)
                ax2.set_xlabel('Character embedding dimension 1')
                ax2.set_ylabel('Character embedding dimension 2')
                ax2.set_title('Mixed model character embeddings')
                plt.colorbar(scatter, ax=ax2, label='Agent ID')
                break
    
    plt.tight_layout()
    plt.savefig('figure3d_mixed_species.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_mixed_species_results(results)

## Summary Statistics

In [ ]:
def print_summary_statistics(results):
    """Print summary statistics for all models"""
    print("=== FIGURE 3 SUMMARY STATISTICS ===")
    print()
    
    for model_name, model_results in results.items():
        print(f"Model: {model_name}")
        print("-" * 40)
        
        for dataset_name, dataset_results in model_results.items():
            if dataset_name == 'bayes_optimal_baseline':
                continue
                
            print(f"  Dataset: {dataset_name}")
            print(f"    Action Accuracy: {dataset_results['action_accuracy']:.3f}")
            print(f"    Mean KL Divergence: {dataset_results['mean_kl_divergence']:.3f}")
            
            # Performance by n_past
            acc_by_n_past = dataset_results['accuracy_by_n_past']
            if acc_by_n_past:
                min_n_past = min(acc_by_n_past.keys())
                max_n_past = max(acc_by_n_past.keys())
                improvement = acc_by_n_past[max_n_past] - acc_by_n_past[min_n_past]
                print(f"    Improvement (n_past={min_n_past} to {max_n_past}): {improvement:.3f}")
            
            print()
        
        print()

print_summary_statistics(results)

## Generate Final Figure 3 Reproduction

In [ ]:
def create_figure3_reproduction(results):
    """Create a single figure reproducing all panels of Figure 3"""
    fig = plt.figure(figsize=(16, 12))
    
    # Panel A: Action likelihood (top left)
    ax1 = plt.subplot(2, 3, 1)
    alpha = 0.01
    model_name = list(results.keys())[0]
    if f'alpha_{alpha}' in results[model_name]:
        accuracy_by_n_past = results[model_name][f'alpha_{alpha}']['accuracy_by_n_past']
        n_past_values = sorted(accuracy_by_n_past.keys())
        accuracies = [accuracy_by_n_past[n] for n in n_past_values]
        ax1.plot(n_past_values, accuracies, 'o-', label='ToMnet', linewidth=2, color='blue')
    
    # Bayes optimal baseline
    if 'bayes_optimal_baseline' in results[model_name]:
        baseline_data = results[model_name]['bayes_optimal_baseline']
        baseline_by_n_past = defaultdict(list)
        for n_past, accuracy in zip(baseline_data['n_past_values'], baseline_data['action_accuracies']):
            baseline_by_n_past[n_past].append(accuracy)
        
        n_past_values = sorted(baseline_by_n_past.keys())
        baseline_accuracies = [np.mean(baseline_by_n_past[n]) for n in n_past_values]
        ax1.plot(n_past_values, baseline_accuracies, 's--', label='Bayes-optimal', 
               color='red', linewidth=2, alpha=0.7)
    
    ax1.set_xlabel('Number of past observations')
    ax1.set_ylabel('Action likelihood')
    ax1.set_title('(a) Near-deterministic agents (α=0.01)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Panel B: Action likelihood for stochastic agents (top middle)
    ax2 = plt.subplot(2, 3, 2)
    alpha = 3.0
    if f'alpha_{alpha}' in results[model_name]:
        accuracy_by_n_past = results[model_name][f'alpha_{alpha}']['accuracy_by_n_past']
        n_past_values = sorted(accuracy_by_n_past.keys())
        accuracies = [accuracy_by_n_past[n] for n in n_past_values]
        ax2.plot(n_past_values, accuracies, 'o-', label='ToMnet', linewidth=2, color='blue')
    
    ax2.set_xlabel('Number of past observations')
    ax2.set_ylabel('Action likelihood')
    ax2.set_title('(b) Stochastic agents (α=3.0)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Panel C: Character embeddings (top right)
    ax3 = plt.subplot(2, 3, 3)
    alpha = 0.01
    if f'alpha_{alpha}' in results[model_name]:
        embeddings = results[model_name][f'alpha_{alpha}']['character_embeddings']
        agent_ids = results[model_name][f'alpha_{alpha}']['agent_ids']
        
        # Use agent ID as proxy for action preference
        scatter = ax3.scatter(embeddings[:, 0], embeddings[:, 1], 
                            c=agent_ids % 5, cmap='tab10', alpha=0.7, s=30)
    
    ax3.set_xlabel('Character embedding dimension 1')
    ax3.set_ylabel('Character embedding dimension 2')
    ax3.set_title('(c) 2D character embeddings')
    
    # Panel D: KL divergence matrix (bottom left)
    ax4 = plt.subplot(2, 3, 4)
    alpha_values = [0.01, 3.0]
    kl_matrix = np.zeros((2, 2))
    
    for i, train_alpha in enumerate(alpha_values):
        for j, test_alpha in enumerate(alpha_values):
            model_key = list(results.keys())[0]  # Use first available model
            if f'alpha_{test_alpha}' in results[model_key]:
                kl_div = results[model_key][f'alpha_{test_alpha}']['mean_kl_divergence']
                kl_matrix[i, j] = kl_div
    
    im = ax4.imshow(kl_matrix, cmap='viridis', aspect='auto')
    ax4.set_xticks([0, 1])
    ax4.set_yticks([0, 1])
    ax4.set_xticklabels(['α=0.01', 'α=3.0'])
    ax4.set_yticklabels(['α=0.01', 'α=3.0'])
    ax4.set_xlabel('Test species')
    ax4.set_ylabel('Training species')
    ax4.set_title('(d) KL divergence matrix')
    
    # Add text annotations
    for i in range(2):
        for j in range(2):
            ax4.text(j, i, f'{kl_matrix[i, j]:.2f}',
                    ha="center", va="center", color="white")
    
    plt.colorbar(im, ax=ax4, shrink=0.6)
    
    # Panel E: Mixed species performance (bottom middle)
    ax5 = plt.subplot(2, 3, 5)
    if 'mixed' in results:
        categories = ['α=0.01 only', 'α=3.0 only', 'Mixed']
        test_alphas = [0.01, 3.0]
        
        # Simplified bar plot
        x = np.arange(len(test_alphas))
        width = 0.3
        
        # Get accuracies for mixed model
        mixed_accs = []
        for test_alpha in test_alphas:
            if f'alpha_{test_alpha}' in results['mixed']:
                acc = results['mixed'][f'alpha_{test_alpha}']['action_accuracy']
                mixed_accs.append(acc)
            else:
                mixed_accs.append(0)
        
        ax5.bar(x, mixed_accs, width, label='Mixed training', color='green')
        ax5.set_xlabel('Test species')
        ax5.set_ylabel('Accuracy')
        ax5.set_title('(e) Mixed species training')
        ax5.set_xticks(x)
        ax5.set_xticklabels([f'α={a}' for a in test_alphas])
        ax5.legend()
    
    # Panel F: Summary plot (bottom right)
    ax6 = plt.subplot(2, 3, 6)
    
    # Create summary comparison
    metrics = ['Accuracy (α=0.01)', 'Accuracy (α=3.0)', 'Mean KL div']
    tomnet_values = []
    
    model_key = list(results.keys())[0]
    for alpha in [0.01, 3.0]:
        if f'alpha_{alpha}' in results[model_key]:
            acc = results[model_key][f'alpha_{alpha}']['action_accuracy']
            tomnet_values.append(acc)
        else:
            tomnet_values.append(0)
    
    # Add mean KL divergence
    mean_kl = np.mean([results[model_key][f'alpha_{alpha}']['mean_kl_divergence'] 
                      for alpha in [0.01, 3.0] if f'alpha_{alpha}' in results[model_key]])
    tomnet_values.append(1 / (1 + mean_kl))  # Convert to "goodness" metric
    
    y_pos = np.arange(len(metrics))
    ax6.barh(y_pos, tomnet_values, color=['blue', 'orange', 'green'])
    ax6.set_yticks(y_pos)
    ax6.set_yticklabels(metrics)
    ax6.set_xlabel('Performance')
    ax6.set_title('(f) Summary metrics')
    
    plt.tight_layout()
    plt.savefig('figure3_reproduction.png', dpi=300, bbox_inches='tight')
    plt.show()

create_figure3_reproduction(results)

## Export Results for Further Analysis

In [ ]:
# Save processed results
processed_results = {
    'action_accuracies': {},
    'kl_divergences': {},
    'character_embeddings': {},
    'summary_stats': {}
}

for model_name, model_results in results.items():
    processed_results['action_accuracies'][model_name] = {}
    processed_results['kl_divergences'][model_name] = {}
    processed_results['character_embeddings'][model_name] = {}
    
    for dataset_name, dataset_results in model_results.items():
        if dataset_name == 'bayes_optimal_baseline':
            continue
            
        processed_results['action_accuracies'][model_name][dataset_name] = dataset_results['action_accuracy']
        processed_results['kl_divergences'][model_name][dataset_name] = dataset_results['mean_kl_divergence']
        processed_results['character_embeddings'][model_name][dataset_name] = {
            'embeddings': dataset_results['character_embeddings'].tolist(),
            'agent_ids': dataset_results['agent_ids'].tolist()
        }

# Save to JSON for easy sharing
import json
with open('figure3_processed_results.json', 'w') as f:
    json.dump(processed_results, f, indent=2)

print("Results exported to figure3_processed_results.json")
print("Figures saved as PNG files")